# MODEL-FORGE: QLoRA SFT + GRPO on the executable-SQL reward (Colab T4)

The on-device path (`python -m app.model_forge.train`, MLX LoRA) is the fast loop;
this notebook is the **cloud scale-up**: QLoRA SFT warm-start, then GRPO where the
reward is *the same executable code* the platform's FORGE eval harness uses
(`grpo_reward.sql_reward` — the SQL must run and return the gold rows).

**You need:** a Colab T4 runtime, and two files from the repo uploaded in cell 2:
`backend/app/model_forge/data_gen.py` and `backend/app/model_forge/grpo_reward.py`.
Every number printed is real: reward means the query executed and matched gold rows.

After training: fuse the adapter for MLX on the Mac (`mlx_lm.fuse`), then either call
`app.model_forge.serving.register_champion(path)` or run a FORGE `model`-kind eval so
promotion goes through the measured-improvement gate.

In [ ]:
%pip install -qU "trl>=0.14" "peft>=0.13" "transformers>=4.46" datasets accelerate bitsandbytes

In [ ]:
from google.colab import files

print("Upload data_gen.py and grpo_reward.py (backend/app/model_forge/)")
uploaded = files.upload()  # writes them into /content
import sys
sys.path.insert(0, "/content")
import importlib
import data_gen, grpo_reward
importlib.reload(data_gen); importlib.reload(grpo_reward)
print("reward contract loaded:", grpo_reward.sql_reward("SELECT 1", [["1"]], conn=None))

In [ ]:
from data_gen import generate_tasks, training_jsonl, SCHEMA_SQL
from pathlib import Path
import json

# same split contract as train.py: held-out set is seed 777 and never trains
eval_tasks = generate_tasks(60, seed=777)
eval_questions = {t.question for t in eval_tasks}
pool = generate_tasks(400, seed=11)
train_tasks = [t for t in pool if t.question not in eval_questions]

rows = training_jsonl(train_tasks)
Path("train.jsonl").write_text("\n".join(json.dumps(r) for r in rows))
print(f"train={len(rows)}  held-out={len(eval_tasks)}  (verifiable, execution-backed)")

## Step 1 — QLoRA SFT warm-start (the stub, filled in)
Teach the base model the *format* (SQL only) before RL. 4-bit Qwen2.5-0.5B-Instruct fits a T4 with room for batch>1.

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                             device_map="auto")

ds = load_dataset("json", data_files="train.jsonl", split="train")
peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
sft = SFTTrainer(
    model=model,
    args=SFTConfig(per_device_train_batch_size=4, gradient_accumulation_steps=4,
                   learning_rate=2e-4, num_train_epochs=2, logging_steps=10,
                   fp16=True, output_dir="sft-out", report_to=[]),
    train_dataset=ds,
    peft_config=peft_cfg,
    processing_class=tok,
)
sft.train()
sft.save_pretrained("sft-adapter")
print("SFT warm-start complete -> sft-adapter")

## Step 2 — GRPO on the executable reward
K candidates per question; rewards come from `sql_reward` (1.0 correct / 0.2 valid-but-wrong / 0.0 broken). TRL's GRPO computes the group-relative advantages from these rewards — the exact objective in `grpo_reward.group_advantages`.

In [ ]:
from data_gen import build_db
from grpo_reward import sql_reward
from trl import GRPOConfig, GRPOTrainer

conn = build_db()  # the same demo DB the platform's eval harness executes against
gold = {t.question: t.expected_rows for t in eval_tasks + train_tasks}

def reward_fn(prompts, completions, question=None, **kwargs):
    out = []
    for q, comp in zip(question, completions):
        text = comp[-1]["content"] if isinstance(comp, list) else comp
        out.append(sql_reward(text, gold.get(q, []), conn=conn))
    return out

grpo_ds = ds.map(
    lambda r: {
        "prompt": r["messages"][:2],          # system + user: the model completes the SQL
        "question": r["messages"][1]["content"].split("Question: ")[-1],
    },
    remove_columns=ds.column_names,
)

trainer = GRPOTrainer(
    model="sft-adapter",
    reward_funcs=[reward_fn],
    args=GRPOConfig(per_device_train_batch_size=8, num_generations=4,
                    max_completion_length=120, learning_rate=1e-5,
                    logging_steps=5, fp16=True, output_dir="grpo-out", report_to=[]),
    train_dataset=grpo_ds,
    processing_class=tok,
)
trainer.train()
trainer.save_model("grpo-adapter")
print("GRPO complete -> grpo-adapter")

## Step 3 — `pass_rate` on the held-out set (the stub, filled in)
Generation must *execute*: a candidate only scores when the SQL runs and returns the gold rows. Compare base vs SFT vs GRPO.

In [ ]:
from data_gen import rows_match
from transformers import AutoModelForCausalLM
from peft import PeftModel

def load_with_adapter(base_or_dir, adapter=None):
    try:
        m = AutoModelForCausalLM.from_pretrained(base_or_dir, torch_dtype=torch.float16,
                                                 device_map="auto")
    except OSError:  # adapter dir: reload base then attach
        return None
    if adapter:
        m = PeftModel.from_pretrained(m, adapter)
    return m

def _generate(m, tok, question: str) -> str:
    messages = [
        {"role": "system", "content": "You write a single SQLite query. Reply with the SQL only, no prose."},
        {"role": "user", "content": f"Schema: {SCHEMA_SQL.strip()}\nQuestion: {question}"},
    ]
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tok(prompt, return_tensors="pt").to(m.device)
    out = m.generate(**ids, max_new_tokens=120, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def pass_rate(model_or_dir, adapter=None, tasks=None):
    """Execute every candidate on the held-out set; return (rate, details).
    A task passes only when its SQL runs AND returns the gold rows."""
    tasks = tasks or eval_tasks
    m = model_or_dir if hasattr(model_or_dir, "generate") else \
        load_with_adapter(model_or_dir, adapter)
    if m is None:
        m = load_with_adapter(BASE, model_or_dir)  # adapter dir passed alone
    m.eval()
    conn = build_db()
    passed, details = 0, []
    for t in tasks:
        try:
            sql = _generate(m, tok, t.question).removeprefix("```sql").removeprefix("```").removesuffix("```").strip()
            got = [list(map(str, r)) for r in conn.execute(sql).fetchall()]
            ok = rows_match(got, t.expected_rows)
        except Exception:
            ok = False
        passed += ok
        details.append({"q": t.question[:50], "passed": ok})
    return passed / len(tasks), details

base_model = load_with_adapter(BASE)
rate_base, _ = pass_rate(base_model)
rate_sft, _ = pass_rate(BASE, adapter="sft-adapter")
rate_grpo, details = pass_rate(BASE, adapter="grpo-adapter")
print(f"base={rate_base:.3f}  sft={rate_sft:.3f}  grpo={rate_grpo:.3f}  (held-out n={len(eval_tasks)})")
print("first 5 misses:", [d["q"] for d in details if not d["passed"]][:5])

## Step 4 — promote through the platform (never around the gate)
1. Download `grpo-adapter/` from Colab.
2. On the Mac: merge for MLX — `python -m mlx_lm fuse --model mlx-community/Qwen2.5-0.5B-Instruct-4bit --save-path data/forge/model --adapter-file grpo-adapter/adapter_model.safetensors` (convert the PEFT adapter first if needed).
3. Prove it with a FORGE `model`-kind eval (`POST /api/forge/evals`, kind=`model`) — promotion happens **only on measured improvement**, then `data/forge/repo` gets the git commit and `serving.register_champion` routes `pvu-sql` at the new weights.